# Week 70, Day 2: Review: CPython

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w070d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import copy

a = [[1, 2], [3]]
b = a
c = copy.copy(a)
d = copy.deepcopy(a)
a[0].append(99)
a.append([4])
print(b)
print(c)
print(d)
print(a is b, a[0] is c[0], a[0] is d[0])

In [ ]:
import gc, weakref

class Node:
    def __init__(self, name):
        self.name = name

def grow(items, extra):
    items += extra          # mutates the caller's list
    extra = None            # only rebinds the local name

data, more = [1], [2, 3]
grow(data, more)
print(data, more)

cache = weakref.WeakValueDictionary()
n = Node("a")
cache["a"] = n
print(list(cache))
del n
gc.collect()
print(list(cache))

t = (1, [2])
t[1].append(3)
print(t)

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive, not {value}")
        obj.__dict__[self.name] = value

class Plugin:
    registry = {}

    def __init_subclass__(cls, key=None, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[key or cls.__name__.lower()] = cls

class Box(Plugin, key="box"):
    width = Positive()

    def __init__(self, width):
        self.width = width

class Crate(Plugin):
    pass

b = Box(3)
print(b.width, b.__dict__, sorted(Plugin.registry))
try:
    b.width = -1
except ValueError as e:
    print(e)
print(type(Box), type(type), isinstance(Box.width, Positive))

In [ ]:
import sys, types, importlib.abc, importlib.util

def area(width, height=1):
    scale = 2
    return width * height * scale + len("xy")

code = area.__code__
print(code.co_argcount, code.co_varnames, "len" in code.co_names, 2 in code.co_consts)

class DictFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def __init__(self, sources):
        self.sources = sources

    def find_spec(self, name, path=None, target=None):
        if name in self.sources:
            return importlib.util.spec_from_loader(name, self)
        return None

    def exec_module(self, module):
        exec(self.sources[module.__name__], module.__dict__)

finder = DictFinder({"greeting": "print('running greeting')\nHELLO = 'hi'"})
sys.meta_path.insert(0, finder)
import greeting
import greeting
print(greeting.HELLO, isinstance(greeting, types.ModuleType))
sys.meta_path.remove(finder)
del sys.modules["greeting"]

> **Key idea.** Names refer to objects and assignment never copies; mutation is visible through every reference. Objects are freed when their reference count reaches zero, with gc for cycles; weak references don't keep objects alive. copy is shallow, deepcopy recursive with a memo. Attribute lookup: data descriptors, then the instance dict, then non-data descriptors and class attributes, then __getattr__. Descriptors store per-instance values in the instance; class decorators, __init_subclass__ and metaclasses customise classes, simplest first. CPython compiles source to code objects of bytecode run by an evaluation loop in frames; imports go through sys.modules, finders, specs and loaders.

### Exercise: A deep copy with a memo

Write `clone(obj, memo=None)`: a deep copy for lists, dicts (copy keys as they are, values deeply), tuples and anything else (returned as it is, as if immutable). Like `copy.deepcopy`, use a `memo` dict keyed by `id` so that an object referenced twice is copied **once** (the copies share it too), and cycles work. Don't use the `copy` module.

In [ ]:
def clone(obj, memo=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_clone():\n    \"deep copies with sharing and cycles\"\n    shared = [1, 2]\n    data = {\"a\": shared, \"b\": shared, \"c\": (shared, 3), \"d\": \"text\"}\n    out = clone(data)\n    assert out == data and out is not data and out[\"a\"] is not shared\n    assert out[\"a\"] is out[\"b\"] and out[\"c\"][0] is out[\"a\"]\n    loop = [1]\n    loop.append(loop)\n    copied = clone(loop)\n    assert copied is not loop and copied[1] is copied and copied[0] == 1\n    assert \"import copy\" not in source and \"deepcopy\" not in source")

### Exercise: A validating descriptor

Write a descriptor `Typed(kind, default=None)` that uses `__set_name__` to learn its name, stores each instance's value in the instance's `__dict__`, returns `default` if the instance has no value yet, returns the descriptor itself when looked up on the class, and raises `TypeError` with the message `"name must be kind, not other"` (for example `"age must be int, not str"`) on a wrong type. Then a subclass `Bounded(kind, low, high, default=None)` that also raises `ValueError` outside `low..high`, reusing `Typed`'s checks via `super()`.

In [ ]:
class Typed:
    ___

class Bounded(Typed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_descriptor():\n    \"descriptors\"\n    class Person:\n        name = Typed(str, \"anon\")\n        age = Bounded(int, 0, 150)\n    p, q = Person(), Person()\n    assert p.name == \"anon\" and p.age is None\n    p.name, p.age = \"Ada\", 36\n    assert (p.name, p.age, q.name) == (\"Ada\", 36, \"anon\") and p.__dict__ == {\"name\": \"Ada\", \"age\": 36}\n    assert isinstance(Person.age, Bounded)\n    try:\n        p.age = \"old\"\n        assert False\n    except TypeError as e:\n        assert str(e) == \"age must be int, not str\"\n    try:\n        p.age = 200\n        assert False\n    except ValueError:\n        assert p.age == 36")

### Exercise: A recording proxy

Write a class `Watched(target)`: attribute **reads** that aren't found on the proxy are delegated to `target` (with `__getattr__`) and their names appended to `self.reads`; attribute **writes** go to the target and are recorded as `("set", name)` in `self.writes`. The proxy's own attributes (`_target`, `reads`, `writes`) must be stored on the proxy without being recorded: set them with `object.__setattr__` in `__init__`.

In [ ]:
class Watched:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_proxy():\n    \"__getattr__ and __setattr__\"\n    class Account:\n        def __init__(self):\n            self.balance = 10\n        def deposit(self, n):\n            self.balance += n\n    acct = Account()\n    w = Watched(acct)\n    w.deposit(5)\n    assert w.balance == 15 and w.reads == [\"deposit\", \"balance\"]\n    w.owner = \"Ada\"\n    assert acct.owner == \"Ada\" and w.writes == [(\"set\", \"owner\")] and \"owner\" not in vars(w)\n    assert vars(w).keys() == {\"_target\", \"reads\", \"writes\"}")

### Exercise: Registering classes

Write a class decorator `register(kind)` that adds the decorated class to a module-level dict `REGISTRY` under `kind` and sets `cls.kind = kind`, returning the class unchanged otherwise; registering the same kind twice raises `KeyError`. Then a base class `Shape` whose `__init_subclass__` checks that every subclass defines an `area` method, raising `TypeError(f"{cls.__name__} has no area")` if not.

In [ ]:
REGISTRY = {}

def register(kind):
    ___

class Shape:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_registry():\n    \"class decorators and __init_subclass__\"\n    REGISTRY.clear()\n    @register(\"sq\")\n    class Square(Shape):\n        def __init__(self, s):\n            self.s = s\n        def area(self):\n            return self.s ** 2\n    assert REGISTRY == {\"sq\": Square} and Square.kind == \"sq\" and Square(3).area() == 9\n    try:\n        @register(\"sq\")\n        class Other(Shape):\n            def area(self):\n                return 0\n        assert False\n    except KeyError:\n        pass\n    try:\n        class Blob(Shape):\n            pass\n        assert False\n    except TypeError as e:\n        assert str(e) == \"Blob has no area\"")

### Exercise: Reading a code object

Write `describe(func)`: a dict with `"params"` (the parameter names, in order, from `co_varnames` and `co_argcount`), `"locals"` (the other local variable names, in order), and `"globals"` (the sorted names in `co_names` that are **not** attributes of anything: approximate this as the names that appear in `func.__globals__` or in `builtins`). Use only the code object, `func.__globals__` and `builtins`, not `inspect`.

In [ ]:
import builtins

def describe(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_describe():\n    \"code objects\"\n    SCALE = 3\n    g = {\"SCALE\": SCALE}\n    exec(\"def f(a, b):\\n    total = a + b\\n    items = sorted([total])\\n    return items[0] * SCALE + len(str(total)).bit_length()\", g)\n    d = describe(g[\"f\"])\n    assert d[\"params\"] == [\"a\", \"b\"] and d[\"locals\"] == [\"total\", \"items\"]\n    assert d[\"globals\"] == [\"SCALE\", \"len\", \"sorted\", \"str\"]\n    def h():\n        return 1\n    assert describe(h) == {\"params\": [], \"locals\": [], \"globals\": []}")

### Bug hunt: One row, three times

`[[0] * 3] * 3` puts the **same** row in the grid three times, so setting one cell changes a whole column: this prints `[[1, 0, 0], [1, 0, 0], [1, 0, 0]]`. Make three separate rows so it prints `[[1, 0, 0], [0, 0, 0], [0, 0, 0]]`.

In [ ]:
grid = [[0] * 3] * 3
grid[0][0] = 1
print(grid)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate rows\"\n    expect_output(\"[[1, 0, 0], [0, 0, 0], [0, 0, 0]]\")")

### Bug hunt: A setter that calls itself

The setter assigns to `self.celsius`, which calls the setter again, forever, until `RecursionError`. Store the value under a different name (`self._celsius`) in both the getter and the setter so this prints `21 69.8`.

In [ ]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self.celsius

    @celsius.setter
    def celsius(self, value):
        self.celsius = value

    @property
    def fahrenheit(self):
        return round(self.celsius * 9 / 5 + 32, 1)

r = Reading(21)
print(r.celsius, r.fahrenheit)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a backing attribute\"\n    expect_output(\"21 69.8\")")

### Bug hunt: A descriptor shared by everyone

The descriptor keeps the value on **itself**, and there's only one descriptor for the whole class, so the second account overwrites the first: this prints `20 20`. Store each value in the instance's `__dict__` (under the descriptor's name) so this prints `10 20`.

In [ ]:
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, owner=None):
        return self.value

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(self.name)
        self.value = value

class Account:
    balance = NonNegative()

    def __init__(self, balance):
        self.balance = balance

a, b = Account(10), Account(20)
print(a.balance, b.balance)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"per-instance storage\"\n    expect_output(\"10 20\")")

### Puzzle: A lazy attribute

Arrange the lines of a non-data descriptor that computes a value once and stores it in the instance, so later lookups find the instance attribute first. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # a = [1]; b = a; b += [2]; a
q = ___   # s = "hi"; t = s; t += "!"; s
r = ___   # True or False: copy.copy([[1]])[0] is the original inner list
s = ___   # which wins for obj.x: "data descriptor" or "instance dict"
t = ___   # type(type)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2] and q == \"hi\" and r is True, \"p, q and r\"\n    assert s == \"data descriptor\" and t is type, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: tracked models

Build a tiny model layer. A descriptor `Field(kind, default=None)` validates types (raising `TypeError`) and stores values in the instance's `__dict__`. A base class `Model` whose `__init_subclass__` collects the subclass's `Field`s **in definition order** (including inherited ones, parents first) into `cls._fields`, a list of names. `Model.__init__(**values)` sets every field (missing ones take their default; an unknown name raises `TypeError`); `__repr__` gives `Name(a=1, b='x')`; `__eq__` compares type and field values. Each model class also keeps a list of **weak references** (`weakref.ref`) to its instances, and the class method `live()` returns how many of them are still alive. (A `WeakSet` won't do: defining `__eq__` makes instances unhashable.)

In [ ]:
import weakref

class Field:
    ___

class Model:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_models():\n    \"descriptors, __init_subclass__ and weak references together\"\n    import gc\n    class Point(Model):\n        x = Field(int, 0)\n        y = Field(int, 0)\n    class Point3(Point):\n        z = Field(int, 0)\n    p = Point(x=1, y=2)\n    assert repr(p) == \"Point(x=1, y=2)\" and Point3._fields == [\"x\", \"y\", \"z\"]\n    assert repr(Point3(z=5)) == \"Point3(x=0, y=0, z=5)\"\n    assert Point(x=1, y=2) == p and Point(x=1) != p and Point3(x=1, y=2) != p\n    try:\n        Point(x=\"1\")\n        assert False\n    except TypeError:\n        pass\n    try:\n        Point(w=1)\n        assert False\n    except TypeError:\n        pass\n    gc.collect()\n    keep = [Point(), Point()]\n    base = Point.live()\n    keep.pop()\n    gc.collect()\n    assert Point.live() == base - 1\n    q = Point3()\n    assert Point3.live() >= 1")

## Warm-up

`b` is the same list; `c` is a new outer list holding the **same** inner lists; `d` copies all the way down. If you got any line wrong, revisit Week 62 Days 1 and 4.

## The memory model

- **Names refer to objects** (Week 62). Assignment never copies: it binds a name to an object. Passing an argument binds the parameter to the same object, so a function can mutate the caller's list but can't rebind the caller's name.
- Every object has an **identity** (`id`, compared with `is`), a **type** and a **value**. Use `is` only for singletons such as `None`; use `==` for values. Mutable objects (lists, dicts, sets, most instances) can change in place; immutable ones (ints, strings, tuples, frozensets) can't, though a tuple can hold a mutable object.
- **Augmented assignment** mutates mutable objects (`lst += [1]` extends in place) but rebinds immutable ones.
- **Garbage collection** (Week 62 Day 3): CPython frees an object as soon as its **reference count** drops to zero; the cyclic collector (`gc`) finds groups that only refer to each other. `__del__` runs when an object is freed, but don't rely on its timing.
- **Copies** (Day 4): `copy.copy` is shallow (a new container with the same items); `copy.deepcopy` copies recursively, using a **memo** so that shared objects stay shared and cycles don't loop forever. Classes customise it with `__copy__` and `__deepcopy__`.
- **Weak references** (Day 5) refer to an object without keeping it alive: `weakref.ref`, `WeakValueDictionary`, `WeakSet`, `finalize`. They're for caches and registries that mustn't stop things being freed.

## Attribute magic

- **Lookup order** for `obj.x` (Week 63): a **data descriptor** on the class (one with `__set__` or `__delete__`, such as a property) wins; then the instance's `__dict__`; then a **non-data descriptor** or plain class attribute (functions are non-data descriptors, which is how methods bind); then `__getattr__`, called only when everything else fails. `__getattribute__` runs for **every** lookup.
- A **descriptor** is a class attribute with `__get__` (and optionally `__set__`, `__delete__`). `__set_name__(owner, name)` tells it the name it was assigned to. Store per-instance values in the instance's `__dict__`, not on the descriptor, which is shared by all instances.
- **`property`** is a data descriptor built from getter, setter and deleter functions; `functools.cached_property` is a non-data descriptor that stores its result in the instance.
- **`__getattr__`** suits proxies and delegation; **`__setattr__`** intercepts every assignment (use `super().__setattr__` inside it to avoid recursion).
- **Customising classes**: a **class decorator** receives the finished class and returns it (or a replacement); **`__init_subclass__`** runs on the parent whenever a subclass is created; a **metaclass** (a subclass of `type`) controls class creation itself through `__new__`, `__init__` and `__call__`. Prefer the simplest that works: decorator or `__init_subclass__` first, a metaclass only when you must control creation.

## Inside CPython

- **How code runs** (Week 64): source is tokenised, parsed into an **AST**, compiled to a **code object** of **bytecode**, and the **evaluation loop** executes it on a stack, one instruction at a time. `.pyc` files in `__pycache__` cache compiled modules.
- **Code objects** hold the bytecode and its tables: `co_consts` (constants), `co_names` (global and attribute names), `co_varnames` (locals, parameters first), `co_argcount`. `dis.dis` shows the instructions; their exact names change between Python versions, so read them, don't depend on them.
- **Frames** (Day 3) hold a running function's locals, its code and the current instruction; each call pushes one, and tracebacks are a list of them. `sys._getframe` and `inspect` let you look at them.
- **Objects in C** (Day 4): every object starts with a `PyObject` header, a reference count and a pointer to its type. Small ints and some strings are cached, which is why `is` sometimes "works" on them: never rely on it.
- **Imports** (Day 5): `import m` checks `sys.modules` first, then asks each **finder** on `sys.meta_path` for a **spec**; the spec's **loader** creates and executes the module, which is stored in `sys.modules` **before** its code runs (which is how circular imports partly work). A module's code runs only on its first import.

## Classic mistakes

- **`[[0] * 3] * 3`** makes three references to **one** row.
- **A mutable default argument** (`def f(items=[])`) is created once, when the function is defined, and shared by every call.
- **A property setter that assigns to the property itself** (`self.x = v` inside the setter for `x`) recurses forever; store the value under another name.
- **Storing per-instance data on a descriptor** (`self.value = v` in `__set__`) shares it across all instances.
- **`is` for values**: `x is 1000` or `name is "text"` depends on caching details. Use `==`.
- **Expecting an import to re-run a module**: it's cached in `sys.modules`; use `importlib.reload` deliberately.

## Practice

## Quiz

## Daily challenge

## Before you go

Add the ideas from today you were least sure of to yesterday's list. Tomorrow: the last review block, performance, concurrency and extending Python.